In [2]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [4]:
data = pd.read_csv('dataset/customer_churn.csv')

In [5]:
x= data.drop(columns= ['RowNumber' ,'CustomerId','Surname' ,'Exited'])
y = data['Exited']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(x , y , test_size= 0.2 , random_state = 42)

In [7]:
from preprocessing import preprocess
preprocessor = preprocess()
X_train= preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)


In [8]:
class customDataset(Dataset):
    def __init__(self, features , labels):
        self.features = torch.tensor(features , dtype = torch.float32)
        self.labels = torch.tensor(labels.values, dtype = torch.float32)
        
        
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self,idx):
        return self.features[idx] , self.labels[idx]
    
    

In [9]:
train_dataset = customDataset(X_train , y_train)
test_dataset = customDataset(X_test, y_test)

In [10]:
train_loader = DataLoader(train_dataset ,batch_size = 64 , shuffle = True , pin_memory = True ,num_workers = 0 )
test_loader = DataLoader(test_dataset ,batch_size = 64 , shuffle = False , pin_memory = True ,num_workers = 0)

In [11]:
class mymodel(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(features ,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p = 0.3),
            nn.Linear(128 , 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p = 0.3),
            nn.Linear(64 , 1),
            # nn.Sigmoid()
            
            
        )
    def forward(self , x):
        return self.model(x)
    

In [12]:
model = mymodel(features = X_train.shape[1])
model.to(device)

neg_count = (y_train == 0).sum()
pos_count = (y_train ==1).sum()
pos_weight = torch.tensor([neg_count / pos_count] , dtype = torch.float32).to(device)

print(f"Neg: {neg_count}, Pos: {pos_count}, Ratio: {neg_count/pos_count:.2f}")

loss = nn.BCEWithLogitsLoss(pos_weight = pos_weight)
optimizer = optim.Adam(model.parameters() , lr = 0.0001 , weight_decay = 1e-4)


Neg: 6355, Pos: 1645, Ratio: 3.86


In [13]:
epochs = 50

for epoch in range(epochs):
    total_loss = 0
    for batch_features , batch_labels in train_loader:
        batch_features=batch_features.to(device)
        batch_labels= batch_labels.to(device)
        output = model(batch_features)
        findloss = loss(output.squeeze() , batch_labels)
        optimizer.zero_grad()
        findloss.backward()
        optimizer.step()
        
        total_loss += findloss.item()
        
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch : {epoch + 1}  Loss: {avg_loss:.4f}")
        
        
    

Epoch : 1  Loss: 1.0042
Epoch : 2  Loss: 0.8436
Epoch : 3  Loss: 0.7087
Epoch : 4  Loss: 0.5839
Epoch : 5  Loss: 0.4646
Epoch : 6  Loss: 0.3643
Epoch : 7  Loss: 0.2947
Epoch : 8  Loss: 0.2368
Epoch : 9  Loss: 0.1961
Epoch : 10  Loss: 0.1658
Epoch : 11  Loss: 0.1412
Epoch : 12  Loss: 0.1214
Epoch : 13  Loss: 0.1018
Epoch : 14  Loss: 0.0884
Epoch : 15  Loss: 0.0827
Epoch : 16  Loss: 0.0709
Epoch : 17  Loss: 0.0669
Epoch : 18  Loss: 0.0569
Epoch : 19  Loss: 0.0547
Epoch : 20  Loss: 0.0478
Epoch : 21  Loss: 0.0455
Epoch : 22  Loss: 0.0410
Epoch : 23  Loss: 0.0385
Epoch : 24  Loss: 0.0396
Epoch : 25  Loss: 0.0353
Epoch : 26  Loss: 0.0310
Epoch : 27  Loss: 0.0320
Epoch : 28  Loss: 0.0287
Epoch : 29  Loss: 0.0308
Epoch : 30  Loss: 0.0311
Epoch : 31  Loss: 0.0265
Epoch : 32  Loss: 0.0246
Epoch : 33  Loss: 0.0247
Epoch : 34  Loss: 0.0224
Epoch : 35  Loss: 0.0230
Epoch : 36  Loss: 0.0241
Epoch : 37  Loss: 0.0215
Epoch : 38  Loss: 0.0207
Epoch : 39  Loss: 0.0200
Epoch : 40  Loss: 0.0201
Epoch : 4

In [14]:
model.eval()

X_test_tensor  = torch.tensor(X_test,  dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test.values,  dtype=torch.float32)
with torch.no_grad():
    output = model(X_test_tensor.to(device))
    predicted = (torch.sigmoid(output).squeeze() > 0.5).float()
    y_true = y_test_tensor.cpu().numpy()
    y_pred = predicted.cpu().numpy()

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00      1607
         1.0       1.00      1.00      1.00       393

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

[[1606    1]
 [   1  392]]


In [15]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test,  dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test.values,  dtype=torch.float32)

In [16]:
# 1. Check if model is just predicting everything as one class
print("Predicted 0s:", (y_pred == 0).sum())
print("Predicted 1s:", (y_pred == 1).sum())

# 2. Check train vs test loss gap
model.eval()
with torch.no_grad():
    train_output = model(X_train_tensor.to(device))
    train_loss = loss(train_output.squeeze(), y_train_tensor.to(device))
    test_output = model(X_test_tensor.to(device))
    test_loss = loss(test_output.squeeze(), y_test_tensor.to(device))

print(f"Train Loss: {train_loss:.6f}")
print(f"Test Loss:  {test_loss:.6f}")

Predicted 0s: 1607
Predicted 1s: 393
Train Loss: 0.010412
Test Loss:  0.020180


In [17]:
# def predict_customer(customer_dict):
#     # customer_dict is one row of raw data
#     df = pd.DataFrame([customer_dict])
    
#     # Apply same preprocessor
#     processed = preprocessor.transform(df)
#     tensor = torch.tensor(processed, dtype=torch.float32).to(device)
    
#     model.eval()
#     with torch.no_grad():
#         output = torch.sigmoid(model(tensor))
#         probability = output.item()
#         prediction = 1 if probability > 0.5 else 0
    
#     return {
#         "churn": bool(prediction),
#         "probability": round(probability, 4)
#     }

# # Example usage
# customer = {
#     'CreditScore': 600, 'Geography': 'France', 'Gender': 'Male',
#     'Age': 40, 'Tenure': 3, 'Balance': 60000.0, 'NumOfProducts': 2,
#     'HasCrCard': 1, 'IsActiveMember': 0, 'EstimatedSalary': 50000.0,
#     'Satisfaction Score': 3, 'Point Earned': 400, 'Card Type': 'GOLD'
# }

# print(predict_customer(customer))

In [21]:
torch.cuda.empty_cache()
import mlflow
import mlflow.pytorch

# Set experiment name
mlflow.set_experiment("bank-churn-prediction")

with mlflow.start_run(run_name="churn-bcewithlogits-adam"):

    # Log hyperparameters
    mlflow.log_params({
        "epochs": epochs,
        "batch_size": 64,
        "learning_rate": 0.0001,
        "dropout": 0.3,
        "optimizer": "Adam",
        "loss_fn": "BCEWithLogitsLoss",
        "hidden_dims": "128-64",
        "pos_weight": round((neg_count / pos_count).item(), 2)
    })

    # Training loop
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for batch_features, batch_labels in train_loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            optimizer.zero_grad()
            output = model(batch_features)
            findloss = loss(output.squeeze(), batch_labels)
            findloss.backward()
            optimizer.step()
            total_loss += findloss.item()

        avg_loss = total_loss / len(train_loader)

        # Validation loss per epoch
        model.eval()
        with torch.no_grad():
            val_output = model(X_test_tensor.to(device))
            val_loss = loss(val_output.squeeze(), y_test_tensor.to(device))

        # Log metrics per epoch
        mlflow.log_metrics({
            "train_loss": avg_loss,
            "val_loss": val_loss.item()
        }, step=epoch)

        print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_loss:.4f} Val Loss: {val_loss:.4f}")

    # Evaluation
    model.eval()
    with torch.no_grad():
        output = model(X_test_tensor.to(device))
        predicted = (torch.sigmoid(output).squeeze() > 0.5).float()
        y_true = y_test_tensor.cpu().numpy()
        y_pred = predicted.cpu().numpy()

    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

    accuracy  = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall    = recall_score(y_true, y_pred)
    f1        = f1_score(y_true, y_pred)

    # Log final metrics
    mlflow.log_metrics({
        "accuracy":  accuracy,
        "precision": precision,
        "recall":    recall,
        "f1_score":  f1
    })

    print(f"\nAccuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")

    # Save and log model
    torch.save(model.state_dict(), "churn_model.pth")
    mlflow.pytorch.log_model(
    model,
    "model",
    serialization_format="pickle"
)


    # Save and log preprocessor
    import joblib
    joblib.dump(preprocessor, "preprocessor.pkl")
    mlflow.log_artifact("preprocessor.pkl")
    mlflow.log_artifact("churn_model.pth")

    print("\nRun logged successfully in MLflow")

Epoch [1/50] Train Loss: 0.0054 Val Loss: 0.0307
Epoch [2/50] Train Loss: 0.0057 Val Loss: 0.0302
Epoch [3/50] Train Loss: 0.0062 Val Loss: 0.0301
Epoch [4/50] Train Loss: 0.0054 Val Loss: 0.0316
Epoch [5/50] Train Loss: 0.0054 Val Loss: 0.0306
Epoch [6/50] Train Loss: 0.0069 Val Loss: 0.0305
Epoch [7/50] Train Loss: 0.0058 Val Loss: 0.0316
Epoch [8/50] Train Loss: 0.0051 Val Loss: 0.0309
Epoch [9/50] Train Loss: 0.0058 Val Loss: 0.0314
Epoch [10/50] Train Loss: 0.0076 Val Loss: 0.0313
Epoch [11/50] Train Loss: 0.0062 Val Loss: 0.0307
Epoch [12/50] Train Loss: 0.0056 Val Loss: 0.0304
Epoch [13/50] Train Loss: 0.0057 Val Loss: 0.0304
Epoch [14/50] Train Loss: 0.0062 Val Loss: 0.0310
Epoch [15/50] Train Loss: 0.0048 Val Loss: 0.0313
Epoch [16/50] Train Loss: 0.0045 Val Loss: 0.0311
Epoch [17/50] Train Loss: 0.0053 Val Loss: 0.0321
Epoch [18/50] Train Loss: 0.0052 Val Loss: 0.0317
Epoch [19/50] Train Loss: 0.0046 Val Loss: 0.0312
Epoch [20/50] Train Loss: 0.0042 Val Loss: 0.0311
Epoch [21

2026/07/01 13:15:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/01 13:15:49 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/07/01 13:15:49 WARNING mlflow.utils.requirements_utils: Found torch version (2.6.0+cu124) contains a local version label (+cu124). MLflow logged a pip requirement for this package as 'torch==2.6.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Epoch [50/50] Train Loss: 0.0055 Val Loss: 0.0328

Accuracy:  0.9990
Precision: 0.9975
Recall:    0.9975
F1 Score:  0.9975


2026/07/01 13:15:57 WARNING mlflow.utils.requirements_utils: Found torch version (2.6.0+cu124) contains a local version label (+cu124). MLflow logged a pip requirement for this package as 'torch==2.6.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.



Run logged successfully in MLflow
